In [ ]:
# install these things
!pip install -U \
transformers==4.46.3 \
huggingface_hub==0.26.2 \
accelerate==1.1.1 \
tokenizers==0.20.3 \
safetensors

In [ ]:
# remove any partially downloaded files are there
!rm -rf ~/.cache/huggingface/hub/models--microsoft--Phi-3-mini-4k-instruct

In [ ]:
#torch is the main library of PyTorch, a popular deep learning framework.
#It is used to create, train, and run neural networks, including LLMs
import torch

# import the model , tokenizer and pipeline from transformers library
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")
model = AutoModelForCausalLM.from_pretrained(
"microsoft/Phi-3-mini-4k-instruct",
device_map="auto",
torch_dtype="auto",
trust_remote_code=True,)

# Create a pipeline (interface to work with models)
generator = pipeline(
    # Tells to generate the text (task type)
"text-generation",
model=model,
tokenizer=tokenizer,
return_full_text=False,
max_new_tokens=50,
do_sample=False,
)
prompt = "Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened."

#  Generates the new tokens with pipeline
output = generator(prompt)

# prints the generated text from the first result returned by the pipeline.
print(output[0]['generated_text'])

In [ ]:
# Prompt
prompt = "The capital of France is"


# Tokenize the input prompt as pytorch tensors and take input ids
input_ids = tokenizer(prompt, return_tensors="pt").input_ids

# Token ids sent to gpu to run
input_ids = input_ids.to("cuda")


# model = complete phi-3 model that we loaded earlier ( complete model has 1.TRANSFORMER PART and 2.LM HEAD PART)
# model = Access the internal Transformer network inside Phi-3.
# input ids are sent to the transformer block
model_output = model.model(input_ids)


# The output of the transformer block are sent to lm head
# lm head converts those vectors into token scores.
# model_output[0] = It contains the final representation of every token.last hidden state
lm_head_output = model.lm_head(model_output[0])

# Now we have final vector
# 0 = selecting the first sentence   , -1 = Last token position
# argmax(-1) = finding the index of the highest score
token_id = lm_head_output[0,-1].argmax(-1)

# the tokenizer has vocabulary with index and words
# Decoding back to text based on index value that mapped to words
tokenizer.decode(token_id)


In [ ]:
print(model_output[0].shape)
print(lm_head_output.shape)

In [ ]:


prompt = "Write a very long email apologizing to Sarah for the tragic gardening mishap. Explain how it happened."
# Tokenize the input prompt
input_ids = tokenizer(prompt, return_tensors="pt").input_ids
input_ids = input_ids.to("cuda")

In [ ]:
# It measures how long the following cell takes to execute.
%%timeit -n 1

# Generate the text
generation_output = model.generate(
input_ids=input_ids,
max_new_tokens=100,

# This controls whether the model reuses previously computed Keys and Values during generation.
# True = Saves the Keys and Values from previous tokens.
use_cache=True
)

In [ ]:
%%timeit -n 1

# Generate the text
generation_output = model.generate(
input_ids=input_ids,
max_new_tokens=100,
use_cache=False) # False = # The model doesnt saves the key values of previous tokens .# At every generation step, the model recomputes all previous tokens:

In [ ]:
# Using True can be faster for generation